# Sample cadastre parcels near individual high-risk polygon boundaries

This notebook randomly selects cadastre parcel representative points near individual risk-polygon boundaries. The default configuration requests **15 points from P01311 and 15 from P01427**, with 20 m inside and outside bands and a fixed random seed.

It reads the outputs of `risk_polygon_household_counts_simple.ipynb` so the selected `risk_polygon_id` values refer to the polygons already reviewed. Each cadastre parcel is represented by one point for sampling. One GeoJSON shows all eligible parcel polygons and flags the randomly selected parcels; another contains only the selected parcel polygons. The requested inside/outside split is as even as possible; if one side has too few candidates, the notebook can take more from the other side while requiring at least one from each. It never samples a parcel twice in the same run.

These are **parcel-based household proxies**, not verified occupied households. The source risk polygons were clipped to alcaldía boundaries, so inspect selected sites for artificial administrative edges before field use. “Outside” means outside the selected polygon; it does not automatically mean outside every other high-risk polygon.

In [ ]:
%pip -q install geopandas pyogrio shapely pyproj rtree

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import geopandas as gpd

try:
    from google.colab import drive
except ImportError:
    drive = None

# Match the output directory used by the baseline notebook.
WORK = Path('/content/drive/MyDrive/Dissertation/data/output')
if drive is not None and not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
if not WORK.exists():
    raise FileNotFoundError(f'Output folder not found: {WORK}')

METRIC_CRS = 'EPSG:32614'
POLYGON_FILE = WORK / 'risk_polygon_household_counts.geojson'
CADASTRE_FILE = WORK / 'catastro_points.geojson'
CADASTRE_PARCELS_FILE = WORK / 'catastro_parcels_4326.geojson'

# Choose 'selected' for the listed polygon IDs or 'all' for every eligible polygon.
POLYGON_MODE = 'selected'
SELECTED_POLYGON_IDS = ['P01311', 'P01427']
SAMPLE_SIZE_PER_POLYGON = 15
INSIDE_BAND_M = 20
OUTSIDE_BAND_M = 20
RANDOM_SEED = 20260928

if POLYGON_MODE not in {'selected', 'all'}:
    raise ValueError("POLYGON_MODE must be 'selected' or 'all'.")
if SAMPLE_SIZE_PER_POLYGON < 2:
    raise ValueError('At least two points per polygon are needed to include both sides.')
if INSIDE_BAND_M <= 0 or OUTSIDE_BAND_M <= 0:
    raise ValueError('Boundary-band widths must be positive metres.')
if len(SELECTED_POLYGON_IDS) != len(set(SELECTED_POLYGON_IDS)):
    raise ValueError('SELECTED_POLYGON_IDS contains duplicates.')

## Load the sampling frame

The risk-polygon GeoJSON contains the `risk_polygon_id` field created by the baseline notebook. The cadastre point GeoJSON contains one representative point per parcel. The baseline also exports the original cadastre parcel polygons. Cadastre point IDs follow saved file order; the export step verifies that each selected point matches the parcel polygon at the same position. Keep these files from the same baseline run when reproducing a sample.

In [ ]:
def load_layer(path, required_columns, allowed_geometry):
    """Read and validate one saved GeoJSON layer, then project it to metres."""
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run the baseline notebook first.')
    layer = gpd.read_file(path)
    missing = set(required_columns) - set(layer.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    if layer.crs is None:
        raise ValueError(f'{path.name} has no CRS.')
    if layer.geometry.isna().any() or layer.geometry.is_empty.any():
        raise ValueError(f'{path.name} contains empty geometries.')
    if allowed_geometry is not None and not layer.geometry.geom_type.isin(allowed_geometry).all():
        raise ValueError(f'{path.name} has unexpected geometry types.')
    return layer.to_crs(METRIC_CRS).reset_index(drop=True)

polygons = load_layer(
    POLYGON_FILE, ['risk_polygon_id', 'district'], {'Polygon', 'MultiPolygon'}
)
cadastre = load_layer(CADASTRE_FILE, ['district'], {'Point'})

polygons['risk_polygon_id'] = polygons['risk_polygon_id'].astype(str).str.strip()
polygons['district'] = polygons['district'].astype(str).str.strip()
cadastre['district'] = cadastre['district'].astype(str).str.strip()
if polygons.risk_polygon_id.duplicated().any():
    duplicates = polygons.loc[polygons.risk_polygon_id.duplicated(), 'risk_polygon_id'].tolist()
    raise ValueError(f'Risk polygon IDs are not unique: {duplicates[:10]}')
if cadastre.empty or polygons.empty:
    raise ValueError('The risk polygon and cadastre inputs must both be non-empty.')

# This identifier follows the persisted cadastre-point file order, not a changing DataFrame index.
cadastre['cadastre_point_id'] = [f'C{i:07d}' for i in range(1, len(cadastre) + 1)]
if POLYGON_MODE == 'selected':
    missing = set(SELECTED_POLYGON_IDS) - set(polygons.risk_polygon_id)
    if missing:
        raise ValueError(f'Selected IDs not found in {POLYGON_FILE.name}: {sorted(missing)}')
    selected_polygons = polygons[polygons.risk_polygon_id.isin(SELECTED_POLYGON_IDS)].copy()
    selected_polygons['_order'] = selected_polygons.risk_polygon_id.map(
        {polygon_id: position for position, polygon_id in enumerate(SELECTED_POLYGON_IDS)}
    )
    selected_polygons = selected_polygons.sort_values('_order').drop(columns='_order')
else:
    selected_polygons = polygons.sort_values('risk_polygon_id').copy()

cadastre_index = cadastre.sindex
print(f'Selected {len(selected_polygons):,} risk polygons and loaded {len(cadastre):,} cadastre points.')
display(selected_polygons[['risk_polygon_id', 'district']].reset_index(drop=True))

## Identify candidates and sample

A candidate must belong to the polygon’s alcaldía and fall within the requested distance of its boundary. Points exactly on the boundary are excluded. The inside and outside sets are therefore distinct.

The sampling function tries an even split. For 15 points, the first polygon targets 8 inside and 7 outside, and the second targets 7 inside and 8 outside. If a side has fewer eligible points, the remaining places move to the other side. The sample still requires at least one point from each side and the full requested total. Previously selected cadastre points are excluded from later polygon draws so the final sample has unique parcels.

In [ ]:
def candidate_points(polygon, district):
    """Return cadastre points within the configured inside/outside boundary bands."""
    search_zone = polygon.boundary.buffer(max(INSIDE_BAND_M, OUTSIDE_BAND_M))
    positions = list(cadastre_index.query(search_zone, predicate='intersects'))
    if not positions:
        return cadastre.iloc[[]].assign(side=pd.Series(dtype='str'), distance_to_boundary_m=pd.Series(dtype='float'))
    candidates = cadastre.iloc[positions].copy()
    candidates = candidates[candidates.district.eq(district)].copy()
    if candidates.empty:
        return candidates.assign(side=pd.Series(dtype='str'), distance_to_boundary_m=pd.Series(dtype='float'))

    candidates['distance_to_boundary_m'] = candidates.geometry.distance(polygon.boundary)
    inside = candidates.geometry.within(polygon) & candidates.distance_to_boundary_m.le(INSIDE_BAND_M)
    outside = candidates.geometry.disjoint(polygon) & candidates.distance_to_boundary_m.le(OUTSIDE_BAND_M)
    candidates = candidates[inside | outside].copy()
    candidates['side'] = np.where(inside.loc[candidates.index], 'inside', 'outside')
    return candidates.sort_values('cadastre_point_id')

def choose_side_counts(inside_available, outside_available, requested, polygon_position):
    """Allocate a near-even sample, shifting shortages while retaining both sides."""
    if inside_available < 1 or outside_available < 1 or inside_available + outside_available < requested:
        return None
    target_inside = requested // 2 + int(requested % 2 == 1 and polygon_position % 2 == 0)
    target_outside = requested - target_inside
    take_inside = min(target_inside, inside_available)
    take_outside = min(target_outside, outside_available)
    remaining = requested - take_inside - take_outside
    if remaining:
        extra_inside = min(remaining, inside_available - take_inside)
        take_inside += extra_inside
        remaining -= extra_inside
    if remaining:
        extra_outside = min(remaining, outside_available - take_outside)
        take_outside += extra_outside
        remaining -= extra_outside
    if remaining or min(take_inside, take_outside) < 1:
        return None
    return take_inside, take_outside

def polygon_rng(seed, polygon_id):
    """Give each polygon a stable random stream independent of Python's hash salt."""
    digest = hashlib.sha256(f'{seed}|{polygon_id}'.encode('utf-8')).digest()
    return np.random.default_rng(int.from_bytes(digest[:8], 'big'))

def draw_points(pool, count, rng):
    """Draw without replacement from a deterministically sorted candidate pool."""
    if count == 0:
        return pool.iloc[[]].copy()
    chosen_positions = rng.choice(len(pool), size=count, replace=False)
    return pool.iloc[chosen_positions].copy()

used_point_ids = set()
sample_parts = []
candidate_parts = []
audit_rows = []
for polygon_position, polygon_row in enumerate(selected_polygons.itertuples(index=False)):
    polygon_id = polygon_row.risk_polygon_id
    candidates = candidate_points(polygon_row.geometry, polygon_row.district)
    if not candidates.empty:
        # Capture eligibility before excluding points chosen for another polygon.
        candidate_parts.append(candidates[['cadastre_point_id', 'side', 'distance_to_boundary_m']].assign(
            risk_polygon_id=polygon_id
        ))
    available = candidates[~candidates.cadastre_point_id.isin(used_point_ids)].copy()
    inside_pool = available[available.side.eq('inside')]
    outside_pool = available[available.side.eq('outside')]
    allocation = choose_side_counts(
        len(inside_pool), len(outside_pool), SAMPLE_SIZE_PER_POLYGON, polygon_position
    )

    audit = {
        'risk_polygon_id': polygon_id,
        'district': polygon_row.district,
        'requested_total': SAMPLE_SIZE_PER_POLYGON,
        'inside_band_m': INSIDE_BAND_M,
        'outside_band_m': OUTSIDE_BAND_M,
        'random_seed': RANDOM_SEED,
        'eligible_inside_before_dedup': int(candidates.side.eq('inside').sum()),
        'eligible_outside_before_dedup': int(candidates.side.eq('outside').sum()),
        'eligible_inside_after_dedup': len(inside_pool),
        'eligible_outside_after_dedup': len(outside_pool),
        'sampled_inside': 0,
        'sampled_outside': 0,
        'status': 'insufficient_candidates' if allocation is None else 'sampled',
    }
    if allocation is not None:
        take_inside, take_outside = allocation
        rng = polygon_rng(RANDOM_SEED, polygon_id)
        picked = pd.concat([
            draw_points(inside_pool, take_inside, rng),
            draw_points(outside_pool, take_outside, rng),
        ], ignore_index=True)
        picked['risk_polygon_id'] = polygon_id
        picked['inside_band_m'] = INSIDE_BAND_M
        picked['outside_band_m'] = OUTSIDE_BAND_M
        picked['random_seed'] = RANDOM_SEED
        picked['sample_rank_within_polygon'] = np.arange(1, len(picked) + 1)
        sample_parts.append(picked)
        used_point_ids.update(picked.cadastre_point_id)
        audit['sampled_inside'] = take_inside
        audit['sampled_outside'] = take_outside
    audit_rows.append(audit)

audit = pd.DataFrame(audit_rows)
display(audit)
audit.to_csv(WORK / 'risk_polygon_boundary_sample_audit.csv', index=False)

if not candidate_parts:
    raise ValueError('No eligible parcels were found; inspect the audit CSV and boundary bands.')
candidate_links = pd.concat(candidate_parts, ignore_index=True)
sample = (
    gpd.GeoDataFrame(pd.concat(sample_parts, ignore_index=True), geometry='geometry', crs=METRIC_CRS)
    if sample_parts else None
)
if sample is not None:
    if sample.cadastre_point_id.duplicated().any():
        raise AssertionError('The sampled cadastre points are not unique.')
    assert sample.groupby('risk_polygon_id').size().eq(SAMPLE_SIZE_PER_POLYGON).all()
    assert sample.groupby('risk_polygon_id').side.nunique().eq(2).all()

## Export and inspect

The first GeoJSON contains **every parcel eligible under the current polygon selection and boundary bands**, one feature per parcel, with its risk-polygon relationships and a `selected_for_sample` flag. The second GeoJSON contains only the selected parcel polygons. Both are WGS 84. The sample CSV includes representative-point coordinates. Before export, the notebook checks that each point matches its parcel polygon; mismatched baseline files cause an error. The audit CSV reports eligible counts and the realised inside/outside split. A fixed seed reproduces the sample when the saved inputs and parameters are unchanged.

In [ ]:
# The baseline writes parcel polygons in the same cadastre-file order as the
# representative points. Validate every exported pair before using that order.
parcels = load_layer(CADASTRE_PARCELS_FILE, [], None)
# The baseline point export omits any non-polygon records in the parcel file.
parcels = parcels[parcels.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)
if len(parcels) != len(cadastre):
    raise ValueError(
        f'{CADASTRE_PARCELS_FILE.name} has {len(parcels):,} parcels but '
        f'{CADASTRE_FILE.name} has {len(cadastre):,} points. '
        'Recreate both files in the same baseline run.'
    )
def with_parcel_geometry(points):
    """Replace representative-point geometry with its verified source parcel polygon."""
    positions = points.cadastre_point_id.str[1:].astype(int).to_numpy() - 1
    matched = parcels.iloc[positions].reset_index(drop=True)
    point_geometry = points.geometry.reset_index(drop=True)
    pair_error_m = matched.geometry.representative_point().distance(point_geometry)
    if pair_error_m.gt(0.05).any():
        bad_ids = points.loc[pair_error_m.gt(0.05).to_numpy(), 'cadastre_point_id'].tolist()
        raise ValueError(
            f'Parcel polygons do not match representative points for {bad_ids[:10]}. '
            'Recreate the cadastre point and parcel outputs in the same baseline run.'
        )
    return gpd.GeoDataFrame(
        points.drop(columns='geometry').reset_index(drop=True),
        geometry=matched.geometry, crs=METRIC_CRS,
    )

# A parcel can qualify for several risk polygons. Keep one feature per parcel
# and retain all polygon IDs separately for its inside and outside relationships.
def joined_ids(values):
    """Join unique polygon IDs in first-seen order for GeoJSON attributes."""
    return ','.join(dict.fromkeys(values))

eligible = cadastre[cadastre.cadastre_point_id.isin(candidate_links.cadastre_point_id)].copy()
eligible = eligible.reset_index(drop=True)
eligible['eligible_risk_polygon_ids'] = eligible.cadastre_point_id.map(
    candidate_links.groupby('cadastre_point_id').risk_polygon_id.agg(joined_ids)
)
for side in ('inside', 'outside'):
    ids_by_point = candidate_links[candidate_links.side.eq(side)].groupby(
        'cadastre_point_id'
    ).risk_polygon_id.agg(joined_ids)
    eligible[f'eligible_{side}_polygon_ids'] = eligible.cadastre_point_id.map(
        ids_by_point
    ).fillna('')
eligible['minimum_boundary_distance_m'] = eligible.cadastre_point_id.map(
    candidate_links.groupby('cadastre_point_id').distance_to_boundary_m.min()
)
eligible['inside_band_m'] = INSIDE_BAND_M
eligible['outside_band_m'] = OUTSIDE_BAND_M
eligible['random_seed'] = RANDOM_SEED

# A selected-mode run with a shortfall has no complete sample to mark as selected.
complete_sample = sample is not None and (
    POLYGON_MODE == 'all' or audit.status.eq('sampled').all()
)
if complete_sample:
    chosen = sample.set_index('cadastre_point_id')
    eligible['selected_for_sample'] = eligible.cadastre_point_id.isin(chosen.index)
    eligible['selected_risk_polygon_id'] = eligible.cadastre_point_id.map(
        chosen.risk_polygon_id
    ).fillna('')
    eligible['selected_side'] = eligible.cadastre_point_id.map(chosen.side).fillna('')
else:
    eligible['selected_for_sample'] = False
    eligible['selected_risk_polygon_id'] = ''
    eligible['selected_side'] = ''
if eligible.cadastre_point_id.duplicated().any():
    raise AssertionError('Eligible output contains duplicate cadastre parcels.')

eligible_wgs84_points = eligible.to_crs(4326)
eligible['representative_point_longitude'] = eligible_wgs84_points.geometry.x.to_numpy()
eligible['representative_point_latitude'] = eligible_wgs84_points.geometry.y.to_numpy()
eligible_parcels = with_parcel_geometry(eligible).to_crs(4326)
eligible_path = WORK / 'risk_polygon_boundary_eligible_parcels.geojson'
eligible_parcels.to_file(eligible_path, driver='GeoJSON')
print(f'Wrote {len(eligible_parcels):,} unique eligible parcels to {eligible_path}')

# Preserve exact sample quotas for named polygons; the candidate layer above
# remains available for diagnosing shortages. All-mode skips short polygons.
if POLYGON_MODE == 'selected' and audit.status.ne('sampled').any():
    failed = audit.loc[audit.status.ne('sampled'), [
        'risk_polygon_id', 'eligible_inside_after_dedup', 'eligible_outside_after_dedup'
    ]]
    raise ValueError(
        'The selected polygons cannot supply the requested distinct inside/outside sample. '
        f'Inspect {eligible_path} and the audit CSV. Failed polygons: {failed.to_dict("records")}'
    )
if sample is None:
    raise ValueError('No polygons supplied a sample; inspect the eligible layer and audit CSV.')

sample_wgs84 = sample.to_crs(4326)
sample_wgs84['sample_point_longitude'] = sample_wgs84.geometry.x
sample_wgs84['sample_point_latitude'] = sample_wgs84.geometry.y
parcel_sample = with_parcel_geometry(sample).to_crs(4326)
parcel_sample['sample_point_longitude'] = sample_wgs84.sample_point_longitude.to_numpy()
parcel_sample['sample_point_latitude'] = sample_wgs84.sample_point_latitude.to_numpy()

output_base = 'risk_polygon_boundary_household_sample'
parcel_sample.to_file(WORK / f'{output_base}.geojson', driver='GeoJSON')
sample_wgs84.drop(columns='geometry').to_csv(WORK / f'{output_base}.csv', index=False)

summary = sample.groupby(['risk_polygon_id', 'side']).size().unstack(fill_value=0)
summary['total'] = summary.sum(axis=1)
display(summary)
print(f'Selected {len(sample):,} distinct cadastre parcels.')
print(f'Wrote {WORK / f"{output_base}.geojson"}')
print(f'Wrote {WORK / f"{output_base}.csv"}')
print(f'Wrote {WORK / "risk_polygon_boundary_sample_audit.csv"}')